In [1]:
# [Célula 1: Setup]
# [Célula 1: Imports e Leitura]
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path
import seaborn as sns
import os




In [ ]:

sns.set_theme(style="whitegrid", palette="dark")

ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")  # Carrega variáveis de ambiente do .env
DIR_DATAPROCESSED = ROOT / os.getenv("DIR_DATAPROCESSED")

df = pd.read_parquet(os.path.join(DIR_DATAPROCESSED, "preprocessado.parquet"))
df = pd.DataFrame(df)

In [5]:
# [Célula 2: Saturação e Oportunidades por UF]
# Calculando a relação entre Volume e número de Competidores por Estado
uf_stats = df.groupby('UF').agg({
    'acessos': 'sum',
    'Empresa': 'nunique'
}).rename(columns={'Empresa': 'Qtd_Competidores'})

# Criando um índice de saturação (Acessos disponíveis por competidor)
uf_stats['Acessos_por_Competidor'] = uf_stats['acessos'] / uf_stats['Qtd_Competidores']
uf_stats = uf_stats.sort_values('Acessos_por_Competidor', ascending=False)

print("--- Ranking de Oportunidades por UF (Menos Saturados) ---")
print("Estados no topo têm alto volume, mas poucos competidores disputando.")
print(uf_stats.head(10))

# [Célula 3: O Avanço das MVNOs (Pequeno Porte)]
# Comparando o comportamento do Pequeno Porte vs Grande Porte
porte_crescimento = df.groupby(['periodo', 'Porte da Prestadora'])['acessos'].sum().unstack()
porte_mom = porte_crescimento.pct_change() * 100

print("\n--- Crescimento MoM por Porte de Empresa (%) ---")
print(porte_mom.tail(3).round(2))

# Em quais produtos o Pequeno Porte está focando?
pequeno_porte = df[df['Porte da Prestadora'] == 'Pequeno Porte']
foco_produtos = pequeno_porte.groupby('Tipo de Produto')['acessos'].sum().sort_values(ascending=False)
foco_produtos_pct = (foco_produtos / foco_produtos.sum()) * 100

print("\n--- Distribuição de Produtos no Pequeno Porte (%) ---")
print(foco_produtos_pct.round(2))

--- Ranking de Oportunidades por UF (Menos Saturados) ---
Estados no topo têm alto volume, mas poucos competidores disputando.
      acessos  Qtd_Competidores  Acessos_por_Competidor
UF                                                     
SP  632472132                19            3.328801e+07
MG  126633001                14            9.045214e+06
PR   93230084                12            7.769174e+06
RJ   90991940                12            7.582662e+06
RS   55571571                13            4.274736e+06
BA   45830452                11            4.166405e+06
GO   34802808                11            3.163892e+06
SC   32249349                13            2.480719e+06
CE   23540507                10            2.354051e+06
PE   23248655                11            2.113514e+06

--- Crescimento MoM por Porte de Empresa (%) ---
Porte da Prestadora  Grande Porte  Pequeno Porte
periodo                                         
2026-01                      1.76           1.16
2026